In [1]:
!pip install statsmodels

In [ ]:
# # =============================================================
# # MIXED-EFFECTS MODEL: Safety Score ~ Visual Attention
# # VERSIÓN CORREGIDA
# # =============================================================

# import pandas as pd
# import numpy as np
# import statsmodels.formula.api as smf
# import statsmodels.api as sm
# from scipy import stats
# import matplotlib.pyplot as plt
# #import warnings
# #warnings.filterwarnings('ignore')

In [3]:
# 0. EDA

In [ ]:
# # 1. CARGA Y PREPARACIÓN DE DATOS

# scores_df = pd.read_csv('precalculated_saliency_coverage.csv')
# gaze_df = pd.read_csv('df_final1.csv')

# print('=== DATOS CARGADOS ===')
# print(f'Scores: {scores_df.shape}')
# print(f'Participantes: {scores_df["participante"].nunique()}')
# print(f'Imágenes: {scores_df["ImageName"].nunique()}')

# # Calcular proporción de atención por clase
# attention_counts = gaze_df.groupby(['participante', 'ImageName', 'main_class']).size().reset_index(name='fixation_count')
# total_fixations = gaze_df.groupby(['participante', 'ImageName']).size().reset_index(name='total')
# attention_counts = attention_counts.merge(total_fixations, on=['participante', 'ImageName'])
# attention_counts['proportion'] = attention_counts['fixation_count'] / attention_counts['total']

# attention_pivot = attention_counts.pivot_table(
#     index=['participante', 'ImageName'],
#     columns='main_class',
#     values='proportion',
#     fill_value=0
# ).reset_index()

# attention_pivot.columns = ['participante', 'ImageName'] + [f'att_{col}' for col in attention_pivot.columns[2:]]

# # Merge datasets
# df = scores_df.merge(attention_pivot, on=['participante', 'ImageName'], how='left').fillna(0)
# print(f'Dataset final: {df.shape}')

In [ ]:
# # 2. CORRELACIONES

# main_vars = ['saliency_coverage', 'stationary_entropy', 'gaze_points_count']
# attention_cols = [col for col in df.columns if col.startswith('att_')]
# top_attention = df[attention_cols].sum().nlargest(10).index.tolist()

# correlations = []
# for col in main_vars + top_attention:
#     if col in df.columns:
#         r, p = stats.pearsonr(df['score'], df[col])
#         correlations.append({'Variable': col, 'r': r, 'p-value': p, 'Significant': p < 0.05})

# corr_df = pd.DataFrame(correlations).sort_values('r', key=abs, ascending=False)
# print('\n=== CORRELACIONES CON SAFETY SCORE ===')
# print(corr_df.to_string(index=False))

In [ ]:
# # 3. PREPARAR MODELO

# top_classes = corr_df[corr_df['Variable'].str.startswith('att_')].head(5)['Variable'].tolist()
# predictors = ['saliency_coverage', 'stationary_entropy', 'gaze_points_count'] + top_classes

# df_model = df.copy()
# for col in predictors:
#     if col in df_model.columns:
#         df_model[f'{col}_z'] = (df_model[col] - df_model[col].mean()) / df_model[col].std()

# z_predictors = [f'{col}_z' for col in predictors if f'{col}_z' in df_model.columns]
# formula = 'score ~ ' + ' + '.join(z_predictors)

# print('\n=== PREDICTORES ===')
# print(predictors)

In [ ]:
# # 4. MIXED-EFFECTS MODEL

# model = smf.mixedlm(formula, data=df_model, groups=df_model['participante'], re_formula='~1')
# result = model.fit(method='lbfgs')
# print(result.summary())


In [ ]:
# # 5. TABLA DE COEFICIENTES (forma segura)

# # Usar summary para extraer coeficientes

# summary_df = pd.DataFrame({
#     'Predictor': result.fe_params.index.tolist(),
#     'Beta': result.fe_params.values,
# })

# # Extraer SE y p-values del summary
# for i, param in enumerate(summary_df['Predictor']):
#     if param in result.bse.index:
#         summary_df.loc[i, 'SE'] = result.bse[param]
#         summary_df.loc[i, 'z'] = result.tvalues[param]
#         summary_df.loc[i, 'p-value'] = result.pvalues[param]

# summary_df['Sig'] = summary_df['p-value'].apply(
#     lambda p: '***' if p<0.001 else ('**' if p<0.01 else ('*' if p<0.05 else '')) if pd.notna(p) else ''
# )
# print(summary_df.to_string(index=False))

In [ ]:
# # 6. ICC (INTRACLASS CORRELATION)

# # Modelo nulo
# null_model = smf.mixedlm('score ~ 1', data=df_model, groups=df_model['participante'])
# null_result = null_model.fit()

# var_participant = float(null_result.cov_re.iloc[0, 0]) if null_result.cov_re.iloc[0, 0] > 0 else 0
# var_residual = null_result.scale
# var_total = var_participant + var_residual

# icc = var_participant / var_total if var_total > 0 else 0

# print(f'Varianza entre participantes: {var_participant:.4f}')
# print(f'Varianza residual: {var_residual:.4f}')
# print(f'ICC = {icc:.4f} ({icc*100:.1f}%)')

# if icc < 0.05:
#     print('\n** ICC ≈ 0: Los participantes califican de forma muy similar.')
#     print('   Esto es BUENO - indica alta concordancia inter-evaluador.')
#     print('   El modelo mixto es válido, pero los efectos aleatorios son mínimos.')

In [ ]:
# # 7. R² Y LIKELIHOOD RATIO TEST (CORREGIDO)

# # Calcular fitted values MANUALMENTE usando solo efectos fijos
# # (evita el error de matriz singular)

# # Obtener coeficientes de efectos fijos
# fixed_effects = result.fe_params

# # Calcular predicciones manualmente
# # y_hat = Intercept + β1*x1 + β2*x2 + ...
# fitted_values = np.full(len(df_model), fixed_effects['Intercept'])

# for predictor in z_predictors:
#     if predictor in fixed_effects.index:
#         fitted_values += fixed_effects[predictor] * df_model[predictor].values

# df_model['fitted'] = fitted_values

# # R² aproximado
# ss_res = ((df_model['score'] - df_model['fitted']) ** 2).sum()
# ss_tot = ((df_model['score'] - df_model['score'].mean()) ** 2).sum()
# r2 = 1 - (ss_res / ss_tot)

# print(f'R² = {r2:.4f}')

# # LRT
# ll_null = null_result.llf
# ll_full = result.llf

# # Verificar que los log-likelihoods sean válidos
# if np.isfinite(ll_null) and np.isfinite(ll_full) and ll_full > ll_null:
#     lr_stat = 2 * (ll_full - ll_null)
#     df_diff = len(result.fe_params) - 1
#     p_value_lrt = 1 - stats.chi2.cdf(lr_stat, df_diff)
#     print(f'LRT: Chi2({df_diff}) = {lr_stat:.2f}, p = {p_value_lrt:.6f}')
    
#     if p_value_lrt < 0.05:
#         print('*** Los predictores mejoran significativamente el modelo ***')
# else:
#     print('LRT: No calculable (log-likelihood infinito o inválido)')
#     print(f'  ll_null = {ll_null}, ll_full = {ll_full}')


In [ ]:
# # 8. VISUALIZACIONES

# # Forest plot (sin random effects)
# fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# # Plot 1: Coeficientes
# coef_plot = summary_df[summary_df['Predictor'] != 'Intercept'].copy()
# y_pos = range(len(coef_plot))

# colors = ['green' if p < 0.05 else 'gray' for p in coef_plot['p-value']]
# axes[0].barh(list(y_pos), coef_plot['Beta'], color=colors, alpha=0.7)
# axes[0].axvline(0, color='red', linestyle='--')
# axes[0].set_yticks(list(y_pos))
# axes[0].set_yticklabels(coef_plot['Predictor'])
# axes[0].set_xlabel('Standardized Beta')
# axes[0].set_title('Fixed Effects (green = p < 0.05)')

# # Plot 2: Residuos
# residuals = df_model['score'] - df_model['fitted']
# axes[1].scatter(df_model['fitted'], residuals, alpha=0.3)
# axes[1].axhline(0, color='red', linestyle='--')
# axes[1].set_xlabel('Fitted Values')
# axes[1].set_ylabel('Residuals')
# axes[1].set_title('Residuals vs Fitted')

# plt.tight_layout()
# plt.savefig('mixed_model_results.png', dpi=150)
# plt.show()



In [ ]:


# sig_predictors = summary_df[(summary_df['p-value'] < 0.05) & (summary_df['Predictor'] != 'Intercept')]

# print(f'''
# STATISTICAL ANALYSIS (Mixed-Effects Model)
# -------------------------------------------
# We conducted linear mixed-effects regression with random intercepts 
# for participants to account for potential rater-level variation.

# Data: N = {len(df_model)} observations
#       {df_model['ImageName'].nunique()} urban images
#       {df_model['participante'].nunique()} raters (10 per image)

# Model: safety_score ~ visual_attention_metrics + (1|participant)

# Results:
# - ICC = {icc:.3f} (low ICC indicates high inter-rater agreement)
# - R² = {r2:.3f}
# - {len(sig_predictors)} significant predictors (p < 0.05)

# Significant predictors:
# ''')

# for _, row in sig_predictors.iterrows():
#     print(f"  - {row['Predictor']}: β = {row['Beta']:.3f}, p < 0.05")

# print('''
# Conclusion: Visual attention patterns significantly predict perceived 
# safety scores, controlling for participant-level variation.
# ''')

# # Guardar resultados
# #summary_df.to_csv('mixed_effects_results.csv', index=False)
# #print('Resultados guardados en: mixed_effects_results.csv') 

In [ ]:
########################################

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import statsmodels.api as sm
from scipy import stats
import matplotlib.pyplot as plt

In [5]:
gaze_df = pd.read_csv('df_final1.csv')
scores_df = pd.read_csv('precalculated_saliency_coverage.csv') 

In [6]:
# 1. PREPARACIÓN DE LAS MÉTRICAS DE ATENCIÓN

# Calculamos la proporción de atención por cada trial
attn_counts = gaze_df.groupby(['participante', 'ImageName', 'main_class']).size().reset_index(name='fix_count')
total_fix = gaze_df.groupby(['participante', 'ImageName']).size().reset_index(name='total_fix')
attn_metrics = attn_counts.merge(total_fix, on=['participante', 'ImageName'])
attn_metrics['attn_prop'] = attn_metrics['fix_count'] / attn_metrics['total_fix']

In [7]:
# 2. PIVOTAR 
df_pivot = attn_metrics.pivot_table(index=['participante', 'ImageName'], 
                                    columns='main_class', values='attn_prop', fill_value=0).reset_index()

In [8]:
df_pivot

main_class,participante,ImageName,ashcan,awning,bag,bench,bicycle,box,bridge,building,...,stoplight,streetlight,table,trade,tree,truck,van,wall,water,windowpane
0,1,3,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.299517,...,0.000000,0.0,0.0,0.0,0.135266,0.0,0.000000,0.000000,0.000000,0.0
1,1,4,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.606667,...,0.000000,0.0,0.0,0.0,0.020000,0.0,0.000000,0.000000,0.000000,0.0
2,1,5,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.115207,...,0.000000,0.0,0.0,0.0,0.000000,0.0,0.000000,0.009217,0.232719,0.0
3,1,7,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.053333,...,0.000000,0.0,0.0,0.0,0.040000,0.0,0.000000,0.391111,0.000000,0.0
4,1,8,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.257463,...,0.000000,0.0,0.0,0.0,0.049751,0.0,0.195274,0.211443,0.000000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1475,30,141,0.0,0.0,0.0,0.0,0.0,0.0,0.312639,0.028825,...,0.000000,0.0,0.0,0.0,0.002217,0.0,0.000000,0.090909,0.000000,0.0
1476,30,142,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.569845,...,0.000000,0.0,0.0,0.0,0.000000,0.0,0.004435,0.037694,0.000000,0.0
1477,30,144,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.565410,...,0.000000,0.0,0.0,0.0,0.008869,0.0,0.000000,0.000000,0.000000,0.0
1478,30,148,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.477435,...,0.011876,0.0,0.0,0.0,0.114014,0.0,0.000000,0.000000,0.000000,0.0


In [9]:
df_pivot.isnull().sum()

main_class
participante    0
ImageName       0
ashcan          0
awning          0
bag             0
bench           0
bicycle         0
box             0
bridge          0
building        0
bus             0
car             0
column          0
conveyor        0
door            0
fence           0
field           0
floor           0
flowerpot       0
fountain        0
grass           0
ground          0
handrail        0
hill            0
house           0
lake            0
motorcycle      0
mountain        0
palm            0
path            0
pedestal        0
person          0
plant           0
pole            0
poster          0
railing         0
road            0
rock            0
sea             0
sidewalk        0
signboard       0
sky             0
stair           0
stairs          0
stoplight       0
streetlight     0
table           0
trade           0
tree            0
truck           0
van             0
wall            0
water           0
windowpane      0
dtype: int64

In [10]:
# 3. INTEGRACIÓN CON SCORES
df_final = scores_df.merge(df_pivot, on=['participante', 'ImageName'], how='inner')

df_model = df_final.dropna().reset_index(drop=True)

In [11]:
# 4. SELECCIÓN DE PREDICTORES Y ESTANDARIZACIÓN (Z-SCORE)
# ---------------------------------------------------------

clases_interes = ['ashcan', 'awning', 'bag',
       'bench', 'bicycle', 'box', 'bridge', 'building', 'bus', 'car', 'column',
       'conveyor', 'door', 'fence', 'field', 'floor', 'flowerpot', 'fountain',
       'grass', 'ground', 'handrail', 'hill', 'house', 'lake', 'motorcycle',
       'mountain', 'palm', 'path', 'pedestal', 'person', 'plant', 'pole',
       'poster', 'railing', 'road', 'rock', 'sea', 'sidewalk', 'signboard',
       'sky', 'stair', 'stairs', 'stoplight', 'streetlight', 'table', 'trade',
       'tree', 'truck', 'van', 'wall', 'water', 'windowpane']

predictores_disponibles = []
for clase in clases_interes:
    if clase in df_final.columns:
        col_std = f'{clase}_std'
        # Estandarizamos para poder comparar impactos (Beta weights)
        df_final[col_std] = stats.zscore(df_final[clase])
        predictores_disponibles.append(col_std)

print(f"Predictores estandarizados para el modelo: {predictores_disponibles}")

Predictores estandarizados para el modelo: ['ashcan_std', 'awning_std', 'bag_std', 'bench_std', 'bicycle_std', 'box_std', 'bridge_std', 'building_std', 'bus_std', 'car_std', 'column_std', 'conveyor_std', 'door_std', 'fence_std', 'field_std', 'floor_std', 'flowerpot_std', 'fountain_std', 'grass_std', 'ground_std', 'handrail_std', 'hill_std', 'house_std', 'lake_std', 'motorcycle_std', 'mountain_std', 'palm_std', 'path_std', 'pedestal_std', 'person_std', 'plant_std', 'pole_std', 'poster_std', 'railing_std', 'road_std', 'rock_std', 'sea_std', 'sidewalk_std', 'signboard_std', 'sky_std', 'stair_std', 'stairs_std', 'stoplight_std', 'streetlight_std', 'table_std', 'trade_std', 'tree_std', 'truck_std', 'van_std', 'wall_std', 'water_std', 'windowpane_std']


In [13]:
predictores_disponibles

['ashcan_std',
 'awning_std',
 'bag_std',
 'bench_std',
 'bicycle_std',
 'box_std',
 'bridge_std',
 'building_std',
 'bus_std',
 'car_std',
 'column_std',
 'conveyor_std',
 'door_std',
 'fence_std',
 'field_std',
 'floor_std',
 'flowerpot_std',
 'fountain_std',
 'grass_std',
 'ground_std',
 'handrail_std',
 'hill_std',
 'house_std',
 'lake_std',
 'motorcycle_std',
 'mountain_std',
 'palm_std',
 'path_std',
 'pedestal_std',
 'person_std',
 'plant_std',
 'pole_std',
 'poster_std',
 'railing_std',
 'road_std',
 'rock_std',
 'sea_std',
 'sidewalk_std',
 'signboard_std',
 'sky_std',
 'stair_std',
 'stairs_std',
 'stoplight_std',
 'streetlight_std',
 'table_std',
 'trade_std',
 'tree_std',
 'truck_std',
 'van_std',
 'wall_std',
 'water_std',
 'windowpane_std']

In [ ]:
# 5. MODELO DE EFECTOS MIXTOS 
# ---------------------------------------------------------
if len(predictores_disponibles) > 0:
    # Eliminamos cualquier fila con NaNs y reseteamos el índice para evitar el IndexError
    cols_to_use = ['score', 'participante'] + predictores_disponibles
    if 'grupo' in df_final.columns:
        cols_to_use.append('grupo')
        
    df_model = df_final[cols_to_use].dropna().reset_index(drop=True)

    # Construimos la fórmula dinámicamente
    formula = "score ~ " + " + ".join(predictores_disponibles)
    
    if 'grupo' in df_model.columns:
        formula += " + C(grupo)"

    print(f"Ejecutando modelo con {len(df_model)} observaciones...")

    # Ejecutamos el Mixed Linear Model
    # Usamos df_model que tiene el índice reseteado
    model = smf.mixedlm(formula, data=df_model, groups=df_model["participante"])
    result = model.fit()

    print("\n=== RESULTADOS DEL MODELO MIXTO ===")
    print(result.summary())
    
    # Guardar resultados para el paper
    summary_df = result.summary().tables[1]
    summary_df.to_csv("mixed_effects_results.csv")
else:
    print("Error: No se encontraron las clases especificadas en los datos.")

Ejecutando modelo con 1480 observaciones...

=== RESULTADOS DEL MODELO MIXTO ===
          Mixed Linear Model Regression Results
Model:              MixedLM Dependent Variable: score     
No. Observations:   1480    Method:             REML      
No. Groups:         30      Scale:              2.7531    
Min. group size:    48      Log-Likelihood:     -2974.7745
Max. group size:    50      Converged:          Yes       
Mean group size:    49.3                                  
----------------------------------------------------------
                Coef.  Std.Err.   z    P>|z| [0.025 0.975]
----------------------------------------------------------
Intercept        5.107    0.157 32.481 0.000  4.799  5.415
ashcan_std       0.099    0.044  2.254 0.024  0.013  0.185
awning_std       0.088    0.050  1.759 0.079 -0.010  0.187
bag_std          0.048    0.045  1.067 0.286 -0.040  0.136
bench_std       -0.116    0.044 -2.628 0.009 -0.202 -0.029
bicycle_std     -0.011    0.048 -0.225 0.822 